# M1 보수 학습 기준 검증

## tl;dr

- `report_pre_7d` 기준에서 Event 20을 제외한 `main_no_event20`을 메인 학습 후보로 검증한다.
- Event 20과 Event 67을 모두 제외한 `strict_no_event20_event67`도 같이 돌려, 장기 anomaly 민감도를 확인한다.
- 이번 노트북은 모델 배포가 아니라 라벨/window 기준이 최소 모델에서 버티는지 확인하는 실험 로그다.

## Context & Methods

- 입력은 기존 `m1_classification_features.csv`를 사용한다.
- window는 `report_pre_7d` 기준만 사용한다.
- 학습 입력은 M1 공통 10개 센서의 7개 통계 feature 70개로 제한한다.
- 날짜, event id, `substation_id`, coverage 관련 값은 audit/해석용 metadata로만 둔다.
- 모델은 `DummyClassifier`와 `LogisticRegression(class_weight="balanced")`만 비교한다.
- 검증은 `substation_id` 기준 group CV로 수행해 같은 기계실이 train/test에 동시에 들어가지 않게 한다.

### Key Assumptions

- Event 20은 low coverage로 인해 모든 학습 dataset에서 제외한다.
- Event 67은 메인 기준에는 포함하고, strict 기준에서는 제외한다.
- 현재 단계에서는 복잡한 모델 확장보다 기준 안정성 확인을 우선한다.

In [1]:
from __future__ import annotations

from pathlib import Path
import math
import warnings

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)

ROOT = Path.cwd()
INPUT_PATH = ROOT / "07_데이터산출물" / "m1_classification_features.csv"
OUTPUT_DIR = ROOT / "07_데이터산출물"

DATASET_SUMMARY_PATH = OUTPUT_DIR / "m1_conservative_training_dataset_summary.csv"
CV_METRICS_PATH = OUTPUT_DIR / "m1_conservative_training_cv_metrics.csv"
CV_PREDICTIONS_PATH = OUTPUT_DIR / "m1_conservative_training_cv_predictions.csv"
THRESHOLD_METRICS_PATH = OUTPUT_DIR / "m1_conservative_training_threshold_metrics.csv"
FEATURE_IMPORTANCE_PATH = OUTPUT_DIR / "m1_conservative_training_feature_importance.csv"
REPORT_PATH = OUTPUT_DIR / "05_M1_보수학습_보고서.md"

SENSOR_COLUMNS = [
    "outdoor_temperature",
    "s_hc1_supply_temperature",
    "s_hc1_supply_temperature_setpoint",
    "p_hc1_return_temperature",
    "p_net_meter_energy",
    "p_net_meter_volume",
    "p_net_meter_heat_power",
    "p_net_meter_flow",
    "p_net_supply_temperature",
    "p_net_return_temperature",
]
FEATURE_STATS = ["mean", "std", "min", "max", "median", "last_minus_first", "missing_rate"]
FEATURE_COLUMNS = [f"{sensor}__{stat}" for sensor in SENSOR_COLUMNS for stat in FEATURE_STATS]
METADATA_COLUMNS = [
    "sample_id",
    "manufacturer",
    "label",
    "source_file",
    "source_event_id",
    "substation_id",
    "window_start",
    "window_end",
    "window_days",
    "window_policy",
    "report_date",
    "possible_anomaly_start",
    "possible_anomaly_end",
    "training_start",
    "training_end",
    "sample_count",
    "expected_sample_count",
    "coverage_rate",
]

DATASET_RULES = {
    "main_no_event20": {
        "positive_excluded_events": {20},
        "description": "Event 20 제외, Event 67 포함",
    },
    "strict_no_event20_event67": {
        "positive_excluded_events": {20, 67},
        "description": "Event 20과 Event 67 모두 제외",
    },
}
LABEL_TO_BINARY = {"normal": 0, "efd_possible": 1}
BINARY_TO_LABEL = {0: "normal", 1: "efd_possible"}
THRESHOLDS = [0.3, 0.4, 0.5, 0.6]
RANDOM_STATE = 42

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Data

In [2]:
features = pd.read_csv(INPUT_PATH)

assert set(features["manufacturer"].unique()) == {"manufacturer_1"}
assert set(features["label"].unique()) == {"normal", "efd_possible"}
assert set(FEATURE_COLUMNS).issubset(features.columns)
assert len(FEATURE_COLUMNS) == 70
assert features["sample_count"].gt(0).all()

missing_feature_cols = [col for col in FEATURE_COLUMNS if col not in features.columns]
unexpected_learning_cols = [col for col in METADATA_COLUMNS if col in FEATURE_COLUMNS]
assert not missing_feature_cols, missing_feature_cols
assert not unexpected_learning_cols, unexpected_learning_cols

features[["label", "source_event_id", "substation_id", "window_policy", "coverage_rate"]].head()

,label,source_event_id,substation_id,window_policy,coverage_rate
0,efd_possible,1,10,report_date_minus_7d_to_report_date,1.00000
1,efd_possible,3,12,report_date_minus_7d_to_report_date,1.00000
2,efd_possible,40,24,report_date_minus_7d_to_report_date,0.99504
3,efd_possible,52,21,report_date_minus_7d_to_report_date,1.00000
4,efd_possible,67,7,report_date_minus_7d_to_report_date,1.00000


In [3]:
def build_dataset(dataset_id: str, excluded_positive_events: set[int]) -> pd.DataFrame:
    normal = features[features["label"] == "normal"].copy()
    positive = features[
        (features["label"] == "efd_possible")
        & (~features["source_event_id"].isin(excluded_positive_events))
    ].copy()
    dataset = pd.concat([normal, positive], ignore_index=True)
    dataset["dataset_id"] = dataset_id
    return dataset.sort_values(["label", "source_event_id", "sample_id"]).reset_index(drop=True)


datasets = {
    dataset_id: build_dataset(dataset_id, rule["positive_excluded_events"])
    for dataset_id, rule in DATASET_RULES.items()
}

assert len(datasets["main_no_event20"]) == 49
assert (datasets["main_no_event20"]["label"] == "efd_possible").sum() == 14
assert len(datasets["strict_no_event20_event67"]) == 48
assert (datasets["strict_no_event20_event67"]["label"] == "efd_possible").sum() == 13
assert not any(df["source_event_id"].eq(20).any() for df in datasets.values())
assert not datasets["strict_no_event20_event67"]["source_event_id"].eq(67).any()

summary_rows = []
for dataset_id, dataset in datasets.items():
    positive_events = dataset.loc[dataset["label"] == "efd_possible", "source_event_id"].astype(int).tolist()
    summary_rows.append(
        {
            "dataset_id": dataset_id,
            "description": DATASET_RULES[dataset_id]["description"],
            "rows": len(dataset),
            "normal_rows": int((dataset["label"] == "normal").sum()),
            "positive_rows": int((dataset["label"] == "efd_possible").sum()),
            "positive_event_ids": ",".join(map(str, sorted(positive_events))),
            "event20_included": bool(20 in positive_events),
            "event67_included": bool(67 in positive_events),
            "window_policy_values": "|".join(sorted(dataset["window_policy"].dropna().unique())),
            "window_days_min": float(dataset["window_days"].min()),
            "window_days_median": float(dataset["window_days"].median()),
            "window_days_max": float(dataset["window_days"].max()),
            "coverage_min": float(dataset["coverage_rate"].min()),
            "coverage_median": float(dataset["coverage_rate"].median()),
            "coverage_max": float(dataset["coverage_rate"].max()),
            "sample_count_min": int(dataset["sample_count"].min()),
            "sample_count_median": float(dataset["sample_count"].median()),
            "sample_count_max": int(dataset["sample_count"].max()),
            "learning_feature_count": len(FEATURE_COLUMNS),
        }
    )

dataset_summary = pd.DataFrame(summary_rows)
dataset_summary.to_csv(DATASET_SUMMARY_PATH, index=False, encoding="utf-8-sig")
dataset_summary

,dataset_id,description,rows,normal_rows,positive_rows,positive_event_ids,event20_included,event67_included,window_policy_values,window_days_min,window_days_median,window_days_max,coverage_min,coverage_median,coverage_max,sample_count_min,sample_count_median,sample_count_max,learning_feature_count
0,main_no_event20,"Event 20 제외, Event 67 포함",49,35,14,"1,3,7,10,15,40,44,47,49,52,53,57,64,67",False,True,normal_event_start_to_end|report_date_minus_7d...,7.0,7.0,7.0,0.99504,1.0,1.0,1003,1008.0,1008,70
1,strict_no_event20_event67,Event 20과 Event 67 모두 제외,48,35,13,"1,3,7,10,15,40,44,47,49,52,53,57,64",False,False,normal_event_start_to_end|report_date_minus_7d...,7.0,7.0,7.0,0.99504,1.0,1.0,1003,1008.0,1008,70


## Results

In [4]:
def make_models() -> dict[str, Pipeline]:
    return {
        "dummy_most_frequent": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("model", DummyClassifier(strategy="most_frequent")),
            ]
        ),
        "logistic_balanced": Pipeline(
            [
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
                (
                    "model",
                    LogisticRegression(
                        class_weight="balanced",
                        max_iter=2000,
                        solver="liblinear",
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        ),
    }


def positive_scores(model: Pipeline, X: pd.DataFrame) -> np.ndarray:
    estimator = model.named_steps["model"]
    if hasattr(model, "predict_proba"):
        probabilities = model.predict_proba(X)
        classes = list(estimator.classes_)
        if 1 in classes:
            return probabilities[:, classes.index(1)]
        return np.zeros(len(X), dtype=float)
    return model.predict(X).astype(float)


def metric_row(y_true: np.ndarray, y_pred: np.ndarray, y_score: np.ndarray | None) -> dict:
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    if y_score is not None and len(np.unique(y_true)) == 2:
        roc_auc = float(roc_auc_score(y_true, y_score))
    else:
        roc_auc = 0.5
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": roc_auc,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }


def train_and_evaluate(dataset_id: str, dataset: pd.DataFrame):
    X = dataset[FEATURE_COLUMNS].copy()
    y = dataset["label"].map(LABEL_TO_BINARY).astype(int).to_numpy()
    groups = dataset["substation_id"].astype(str).to_numpy()

    splitter = GroupKFold(n_splits=5)
    metric_rows = []
    prediction_rows = []
    importance_rows = []

    for fold, (train_index, test_index) in enumerate(splitter.split(X, y, groups), start=1):
        X_train, X_test = X.iloc[train_index], X.iloc[test_index]
        y_train, y_test = y[train_index], y[test_index]
        train_groups = sorted(set(groups[train_index]))
        test_groups = sorted(set(groups[test_index]))
        assert set(train_groups).isdisjoint(test_groups)

        for model_name, model in make_models().items():
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test).astype(int)
            y_score = positive_scores(model, X_test)
            row = metric_row(y_test, y_pred, y_score)
            row.update(
                {
                    "dataset_id": dataset_id,
                    "model": model_name,
                    "fold": fold,
                    "train_rows": int(len(train_index)),
                    "test_rows": int(len(test_index)),
                    "train_groups": "|".join(train_groups),
                    "test_groups": "|".join(test_groups),
                    "test_normal": int((y_test == 0).sum()),
                    "test_efd_possible": int((y_test == 1).sum()),
                }
            )
            metric_rows.append(row)

            prediction_meta = dataset.iloc[test_index][
                [
                    "dataset_id",
                    "sample_id",
                    "label",
                    "source_event_id",
                    "substation_id",
                    "window_start",
                    "window_end",
                    "window_days",
                    "coverage_rate",
                    "sample_count",
                ]
            ].copy()
            prediction_meta["fold"] = fold
            prediction_meta["model"] = model_name
            prediction_meta["actual_binary"] = y_test
            prediction_meta["predicted_binary"] = y_pred
            prediction_meta["predicted_label"] = [BINARY_TO_LABEL[int(v)] for v in y_pred]
            prediction_meta["positive_score"] = y_score
            prediction_meta["is_correct"] = prediction_meta["actual_binary"].eq(
                prediction_meta["predicted_binary"]
            )
            prediction_rows.extend(prediction_meta.to_dict("records"))

            if model_name == "logistic_balanced":
                coefficients = model.named_steps["model"].coef_[0]
                for feature, coefficient in zip(FEATURE_COLUMNS, coefficients):
                    sensor, stat = feature.split("__", 1)
                    importance_rows.append(
                        {
                            "dataset_id": dataset_id,
                            "fold": fold,
                            "feature": feature,
                            "sensor": sensor,
                            "stat": stat,
                            "coefficient": float(coefficient),
                            "abs_coefficient": float(abs(coefficient)),
                        }
                    )

    return metric_rows, prediction_rows, importance_rows


all_metric_rows = []
all_prediction_rows = []
all_importance_rows = []
for dataset_id, dataset in datasets.items():
    metric_rows, prediction_rows, importance_rows = train_and_evaluate(dataset_id, dataset)
    all_metric_rows.extend(metric_rows)
    all_prediction_rows.extend(prediction_rows)
    all_importance_rows.extend(importance_rows)

cv_metrics = pd.DataFrame(all_metric_rows)
cv_predictions = pd.DataFrame(all_prediction_rows)
fold_importance = pd.DataFrame(all_importance_rows)

cv_metrics.to_csv(CV_METRICS_PATH, index=False, encoding="utf-8-sig")
cv_predictions.to_csv(CV_PREDICTIONS_PATH, index=False, encoding="utf-8-sig")

cv_metrics.groupby(["dataset_id", "model"]).agg(
    accuracy_mean=("accuracy", "mean"),
    balanced_accuracy_mean=("balanced_accuracy", "mean"),
    precision_mean=("precision", "mean"),
    recall_mean=("recall", "mean"),
    f1_mean=("f1", "mean"),
    roc_auc_mean=("roc_auc", "mean"),
    fp_sum=("fp", "sum"),
    fn_sum=("fn", "sum"),
).reset_index()

,dataset_id,model,accuracy_mean,balanced_accuracy_mean,precision_mean,recall_mean,f1_mean,roc_auc_mean,fp_sum,fn_sum
0,main_no_event20,dummy_most_frequent,0.715556,0.500000,0.000000,0.000000,0.000000,0.500000,0,14
1,main_no_event20,logistic_balanced,0.548889,0.513452,0.251429,0.493333,0.322222,0.538714,16,6
2,strict_no_event20_event67,dummy_most_frequent,0.728889,0.500000,0.000000,0.000000,0.000000,0.500000,0,13
3,strict_no_event20_event67,logistic_balanced,0.544444,0.544643,0.313333,0.533333,0.368889,0.588690,16,6


In [5]:
logistic_predictions = cv_predictions[cv_predictions["model"] == "logistic_balanced"].copy()
threshold_rows = []

for dataset_id, dataset_predictions in logistic_predictions.groupby("dataset_id"):
    y_true = dataset_predictions["actual_binary"].to_numpy(dtype=int)
    y_score = dataset_predictions["positive_score"].to_numpy(dtype=float)
    normal_count = int((y_true == 0).sum())
    positive_count = int((y_true == 1).sum())
    for threshold in THRESHOLDS:
        y_pred = (y_score >= threshold).astype(int)
        row = metric_row(y_true, y_pred, y_score)
        row.update(
            {
                "dataset_id": dataset_id,
                "model": "logistic_balanced",
                "threshold": threshold,
                "rows": int(len(y_true)),
                "normal_rows": normal_count,
                "positive_rows": positive_count,
                "false_positive_rate": row["fp"] / normal_count if normal_count else math.nan,
                "false_negative_rate": row["fn"] / positive_count if positive_count else math.nan,
            }
        )
        threshold_rows.append(row)

threshold_metrics = pd.DataFrame(threshold_rows)
threshold_metrics.to_csv(THRESHOLD_METRICS_PATH, index=False, encoding="utf-8-sig")
threshold_metrics

,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,tn,fp,fn,tp,dataset_id,model,threshold,rows,normal_rows,positive_rows,false_positive_rate,false_negative_rate
0,0.551020,0.578571,0.346154,0.642857,0.450000,0.632653,18,17,5,9,main_no_event20,logistic_balanced,0.3,49,35,14,0.485714,0.357143
1,0.551020,0.557143,0.333333,0.571429,0.421053,0.632653,19,16,6,8,main_no_event20,logistic_balanced,0.4,49,35,14,0.457143,0.428571
2,0.551020,0.557143,0.333333,0.571429,0.421053,0.632653,19,16,6,8,main_no_event20,logistic_balanced,0.5,49,35,14,0.457143,0.428571
3,0.612245,0.600000,0.380952,0.571429,0.457143,0.632653,22,13,6,8,main_no_event20,logistic_balanced,0.6,49,35,14,0.371429,0.428571
4,0.562500,0.603297,0.346154,0.692308,0.461538,0.610989,18,17,4,9,strict_no_event20_event67,logistic_balanced,0.3,48,35,13,0.485714,0.307692
5,0.541667,0.540659,0.304348,0.538462,0.388889,0.610989,19,16,6,7,strict_no_event20_event67,logistic_balanced,0.4,48,35,13,0.457143,0.461538
6,0.541667,0.540659,0.304348,0.538462,0.388889,0.610989,19,16,6,7,strict_no_event20_event67,logistic_balanced,0.5,48,35,13,0.457143,0.461538
7,0.583333,0.545055,0.315789,0.461538,0.375000,0.610989,22,13,7,6,strict_no_event20_event67,logistic_balanced,0.6,48,35,13,0.371429,0.538462


In [6]:
feature_importance = fold_importance.groupby(
    ["dataset_id", "feature", "sensor", "stat"], as_index=False
).agg(
    coefficient_mean=("coefficient", "mean"),
    coefficient_std=("coefficient", "std"),
    abs_coefficient_mean=("abs_coefficient", "mean"),
    abs_coefficient_std=("abs_coefficient", "std"),
)
feature_importance["rank_abs_coefficient"] = feature_importance.groupby("dataset_id")[
    "abs_coefficient_mean"
].rank(method="first", ascending=False).astype(int)
feature_importance = feature_importance.sort_values(["dataset_id", "rank_abs_coefficient"])
feature_importance.to_csv(FEATURE_IMPORTANCE_PATH, index=False, encoding="utf-8-sig")

feature_importance.groupby("dataset_id").head(10)

,dataset_id,feature,sensor,stat,coefficient_mean,coefficient_std,abs_coefficient_mean,abs_coefficient_std,rank_abs_coefficient
42,main_no_event20,p_net_return_temperature__last_minus_first,p_net_return_temperature,last_minus_first,-0.458569,0.342419,0.488914,0.285153,1
53,main_no_event20,p_net_supply_temperature__min,p_net_supply_temperature,min,-0.486647,0.128438,0.486647,0.128438,2
56,main_no_event20,s_hc1_supply_temperature__last_minus_first,s_hc1_supply_temperature,last_minus_first,-0.469795,0.399997,0.469795,0.399997,3
0,main_no_event20,outdoor_temperature__last_minus_first,outdoor_temperature,last_minus_first,0.456660,0.192035,0.456660,0.192035,4
25,main_no_event20,p_net_meter_flow__min,p_net_meter_flow,min,-0.383711,0.367979,0.451526,0.254177,5
7,main_no_event20,p_hc1_return_temperature__last_minus_first,p_hc1_return_temperature,last_minus_first,-0.443886,0.173640,0.443886,0.173640,6
60,main_no_event20,s_hc1_supply_temperature__min,s_hc1_supply_temperature,min,-0.427125,0.203970,0.427125,0.203970,7
9,main_no_event20,p_hc1_return_temperature__mean,p_hc1_return_temperature,mean,0.410367,0.071687,0.410367,0.071687,8
50,main_no_event20,p_net_supply_temperature__max,p_net_supply_temperature,max,0.396581,0.218096,0.396581,0.218096,9
11,main_no_event20,p_hc1_return_temperature__min,p_hc1_return_temperature,min,-0.391883,0.229704,0.392095,0.229253,10


## Takeaways

In [7]:
def markdown_table(frame: pd.DataFrame, columns: list[str] | None = None) -> str:
    if columns is not None:
        frame = frame[columns].copy()
    formatted = frame.copy()
    for column in formatted.columns:
        formatted[column] = formatted[column].map(
            lambda value: ""
            if pd.isna(value)
            else f"{value:.4f}"
            if isinstance(value, float)
            else str(value)
        )
    header = "| " + " | ".join(formatted.columns) + " |"
    sep = "| " + " | ".join(["---"] * len(formatted.columns)) + " |"
    rows = ["| " + " | ".join(row) + " |" for row in formatted.astype(str).to_numpy()]
    return "\n".join([header, sep, *rows])


cv_summary = cv_metrics.groupby(["dataset_id", "model"], as_index=False).agg(
    accuracy_mean=("accuracy", "mean"),
    balanced_accuracy_mean=("balanced_accuracy", "mean"),
    precision_mean=("precision", "mean"),
    recall_mean=("recall", "mean"),
    f1_mean=("f1", "mean"),
    roc_auc_mean=("roc_auc", "mean"),
    fp_sum=("fp", "sum"),
    fn_sum=("fn", "sum"),
)
logistic_cv_summary = cv_summary[cv_summary["model"] == "logistic_balanced"].copy()
dummy_cv_summary = cv_summary[cv_summary["model"] == "dummy_most_frequent"][
    ["dataset_id", "balanced_accuracy_mean", "recall_mean", "f1_mean"]
].rename(
    columns={
        "balanced_accuracy_mean": "dummy_balanced_accuracy_mean",
        "recall_mean": "dummy_recall_mean",
        "f1_mean": "dummy_f1_mean",
    }
)
decision_summary = logistic_cv_summary.merge(dummy_cv_summary, on="dataset_id", how="left")
decision_summary["balanced_accuracy_lift_vs_dummy"] = (
    decision_summary["balanced_accuracy_mean"] - decision_summary["dummy_balanced_accuracy_mean"]
)
decision_summary["recall_lift_vs_dummy"] = (
    decision_summary["recall_mean"] - decision_summary["dummy_recall_mean"]
)

threshold_05 = threshold_metrics[threshold_metrics["threshold"].eq(0.5)].set_index("dataset_id")
main_05 = threshold_05.loc["main_no_event20"]
strict_05 = threshold_05.loc["strict_no_event20_event67"]
decision_by_dataset = decision_summary.set_index("dataset_id")
max_lift_vs_dummy = float(decision_summary["balanced_accuracy_lift_vs_dummy"].max())
min_default_false_positive_rate = float(threshold_05["false_positive_rate"].min())

if max_lift_vs_dummy < 0.05 or min_default_false_positive_rate > 0.35:
    conclusion = "두 기준 모두 불안정하므로 positive 샘플 확장 우선"
    conclusion_reason = (
        "Dummy 대비 balanced accuracy 개선폭이 0.05 미만이고, threshold 0.5 기준 false positive가 높다."
    )
elif (
    main_05["balanced_accuracy"] > 0.55
    and main_05["recall"] >= strict_05["recall"]
    and main_05["false_positive_rate"] <= 0.35
):
    conclusion = "`main_no_event20` 기준으로 다음 단계 진행"
    conclusion_reason = (
        "Event 20 제거 후 coverage 문제는 줄었고, Event 67 포함 기준이 strict 기준보다 "
        "recall을 더 잘 유지했다."
    )
elif (
    strict_05["balanced_accuracy"] >= main_05["balanced_accuracy"] - 0.02
    and strict_05["false_positive_rate"] <= main_05["false_positive_rate"]
):
    conclusion = "`strict_no_event20_event67` 기준으로 더 보수적으로 진행"
    conclusion_reason = (
        "Event 67 제외 기준이 성능 손실을 크게 만들지 않으면서 같은 수준 이하의 false positive를 보였다."
    )
else:
    conclusion = "두 기준 모두 불안정하므로 positive 샘플 확장 우선"
    conclusion_reason = "balanced accuracy, recall, false positive tradeoff가 동시에 충분히 안정적이지 않다."

top_features = feature_importance[feature_importance["rank_abs_coefficient"] <= 8].copy()

report = f'''# M1 보수학습 보고서

## 개요

라벨/window 검증 결과를 반영해 `report_pre_7d` 기준의 보수 학습을 수행했다. 이번 단계는 운영 모델 저장이 아니라, Event 20/67 처리 기준에 따라 최소 모델 성능이 얼마나 버티는지 확인하는 것이다.

## 결론

{conclusion}

- 판단 이유: {conclusion_reason}
- 메인 기준은 Event 20을 제외하고 Event 67은 포함한다.
- strict 기준은 Event 20과 Event 67을 모두 제외해 장기 anomaly 민감도를 확인한다.
- 모델 파일은 저장하지 않았다.

## Dataset 요약

{markdown_table(dataset_summary, ["dataset_id", "rows", "normal_rows", "positive_rows", "event20_included", "event67_included", "coverage_min", "coverage_median", "learning_feature_count"])}

## Group CV 요약

{markdown_table(decision_summary, ["dataset_id", "model", "balanced_accuracy_mean", "precision_mean", "recall_mean", "f1_mean", "roc_auc_mean", "fp_sum", "fn_sum", "balanced_accuracy_lift_vs_dummy"])}

## Threshold 0.3~0.6 비교

{markdown_table(threshold_metrics, ["dataset_id", "threshold", "balanced_accuracy", "precision", "recall", "f1", "fp", "fn", "false_positive_rate", "false_negative_rate"])}

## 상위 feature 후보

{markdown_table(top_features, ["dataset_id", "rank_abs_coefficient", "feature", "coefficient_mean", "abs_coefficient_mean"])}

## 해석

- Event 20을 제외하면 coverage 최저값이 개선되어 학습 입력 품질이 안정된다.
- Event 67 포함/제외 비교는 장기 anomaly가 recall과 false positive에 주는 영향을 보는 민감도 분석이다.
- 현재 샘플 수가 작기 때문에 coefficient 순위는 확정 feature가 아니라 반복 검증 후보로만 본다.
- 다음 단계는 특정 기준으로 모델을 확정하기보다 positive 샘플 확장과 라벨 재검토를 먼저 확인하는 것이다.

## 생성 파일

- `07_데이터산출물/m1_conservative_training_dataset_summary.csv`
- `07_데이터산출물/m1_conservative_training_cv_metrics.csv`
- `07_데이터산출물/m1_conservative_training_cv_predictions.csv`
- `07_데이터산출물/m1_conservative_training_threshold_metrics.csv`
- `07_데이터산출물/m1_conservative_training_feature_importance.csv`

## 한계와 주의점

- positive가 13~14건 수준이라 fold별 지표 변동이 크다.
- Logistic coefficient는 표준화된 통계 feature 기준의 방향성 참고값이며, 인과 해석 근거가 아니다.
- 이 결과만으로 운영 threshold를 확정하지 않는다.
'''

report = "\n".join(line.strip() for line in report.splitlines())
REPORT_PATH.write_text(report, encoding="utf-8")
print(conclusion)
print(REPORT_PATH)

두 기준 모두 불안정하므로 positive 샘플 확장 우선
C:\3rd_Project\HeatGridAgent\07_데이터산출물\05_M1_보수학습_보고서.md


In [8]:
output_paths = [
    DATASET_SUMMARY_PATH,
    CV_METRICS_PATH,
    CV_PREDICTIONS_PATH,
    THRESHOLD_METRICS_PATH,
    FEATURE_IMPORTANCE_PATH,
    REPORT_PATH,
]
for path in output_paths:
    assert path.exists(), path

assert len(dataset_summary) == 2
assert dataset_summary.set_index("dataset_id").loc["main_no_event20", "rows"] == 49
assert dataset_summary.set_index("dataset_id").loc["main_no_event20", "positive_rows"] == 14
assert dataset_summary.set_index("dataset_id").loc["strict_no_event20_event67", "rows"] == 48
assert dataset_summary.set_index("dataset_id").loc["strict_no_event20_event67", "positive_rows"] == 13
assert not cv_predictions["source_event_id"].eq(20).any()
assert not cv_predictions[
    cv_predictions["dataset_id"].eq("strict_no_event20_event67")
]["source_event_id"].eq(67).any()
assert len(cv_metrics) == 20
assert len(threshold_metrics) == 8
assert len(feature_importance) == 140
assert set(cv_metrics["model"].unique()) == {"dummy_most_frequent", "logistic_balanced"}

overlap_count = 0
for _, row in cv_metrics.iterrows():
    train_groups = set(str(row["train_groups"]).split("|"))
    test_groups = set(str(row["test_groups"]).split("|"))
    if train_groups & test_groups:
        overlap_count += 1
assert overlap_count == 0

assert set(feature_importance["sensor"].unique()) == set(SENSOR_COLUMNS)
assert set(feature_importance["stat"].unique()) == set(FEATURE_STATS)

{
    "outputs": len(output_paths),
    "cv_metric_rows": len(cv_metrics),
    "prediction_rows": len(cv_predictions),
    "threshold_rows": len(threshold_metrics),
    "feature_importance_rows": len(feature_importance),
}

{'outputs': 6,
 'cv_metric_rows': 20,
 'prediction_rows': 194,
 'threshold_rows': 8,
 'feature_importance_rows': 140}